# Local Inference Comparison: vLLM vs Ollama
## Benchmarks, Setup, and Best Practices

Choose the right backend for your use case:
- **vLLM**: Fast, flexible, production-ready (Recommended)
- **Ollama**: Simple, requires daemon, local only (Fallback)

This notebook demonstrates both backends with performance comparisons.

## Part 1: Environment Detection & Setup

In [ ]:
import os
import sys
import platform
import subprocess
from pathlib import Path

# Detect environment
environment = "Unknown"
is_cloud = False

try:
    import google.colab
    environment = "Google Colab"
    is_cloud = True
except:
    pass

if os.path.exists('/kaggle'):
    environment = "Kaggle Notebooks"
    is_cloud = True
elif os.environ.get('SM_MODEL_DIR'):
    environment = "AWS SageMaker"
    is_cloud = True

if environment == "Unknown":
    environment = f"Local {platform.system()}"

print(f"🔍 Environment: {environment}")
print(f"☁️  Cloud Native: {is_cloud}")
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"💻 Platform: {platform.platform()}")

## Part 2: Installation (Choose One)

In [ ]:
# Option A: vLLM (Recommended for this notebook)
# Uncomment if running in Colab/Kaggle:

# !pip install -q vllm torch

# For local development:
# pip install vllm

In [ ]:
# Option B: Ollama (Local only, requires daemon)
# Installation:
# 1. Download from ollama.ai
# 2. Run: ollama serve (in terminal)
# 3. Pull model: ollama pull mistral
# 4. Install Python client: pip install ollama

# Check if available
try:
    import ollama
    print("✅ Ollama Python client installed")
except ImportError:
    print("❌ Ollama not available. Install: pip install ollama")

## Part 3: Comparison Table

In [ ]:
import pandas as pd

comparison_data = {
    'Feature': [
        'Installation',
        'Local Support',
        'Cloud Support',
        'Speed (tok/s)',
        'Memory Usage',
        'Setup Complexity',
        'Model Selection',
        'Quantization',
        'Production Ready',
        'Price'
    ],
    'vLLM': [
        'pip install vllm',
        '✅ Native',
        '✅ Excellent',
        '100-300',
        'Low (PagedAttention)',
        'Medium',
        'Any HF model',
        'GPTQ, AWQ, KV',
        '✅ Yes',
        'Free (OSS)'
    ],
    'Ollama': [
        'Installer or Docker',
        '✅ Native',
        '❌ Limited',
        '30-50',
        'Higher',
        'Very Low',
        'Limited catalog',
        'Basic (int4, int8)',
        '✅ Yes',
        'Free (OSS)'
    ]
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))
print("\n🎯 Recommendation: Use vLLM for speed, Ollama for simplicity")

## Part 4: Using SAARA's LocalInferenceEngine

In [ ]:
# Add SAARA to path if needed
import sys
sys.path.insert(0, '/path/to/saara_ai-1.6.4')

from saara.local_inference import LocalInferenceEngine, InferenceConfig

print("✅ SAARA LocalInferenceEngine imported successfully")

## Part 5: Quick Start Examples

In [ ]:
# Example 1: Auto-Select Best Backend
print("Example 1: Auto-Select Backend\n")

config = InferenceConfig(
    model="mistral:latest",
    temperature=0.7,
    max_tokens=256
)

try:
    engine = LocalInferenceEngine(config)
    print(f"Backend Selected: {engine.backend_name}")
    print(f
info: {engine.get_info()}\n")

    # Generate response
    prompt = "Explain machine learning in one sentence"
    response = engine.generate(prompt)
    print(f"Q: {prompt}")
    print(f"A: {response}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Have you installed vLLM or Ollama?")

In [ ]:
# Example 2: Streaming Output
print("Example 2: Streaming Output\n")

try:
    prompt = "Write a haiku about AI"
    print(f"Q: {prompt}")
    print("A: ", end="")
    
    for chunk in engine.generate_stream(prompt):
        print(chunk, end="", flush=True)
    print("\n")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# Example 3: System Prompt
print("Example 3: System Prompt\n")

try:
    response = engine.generate(
        prompt="What's 15+27?",
        system_prompt="You are a helpful math tutor. Always show your work."
    )
    print(response)
except Exception as e:
    print(f"❌ Error: {e}")

## Part 6: Performance Benchmarks

In [ ]:
import time

print("Performance Benchmark\n")
print(f"Backend: {engine.backend_name}")
print(f"Model: {engine.config.model}\n")

test_prompts = [
    "Hello",
    "What is AI?",
    "Explain quantum computing in detail and provide examples."
]

results = []
for i, prompt in enumerate(test_prompts, 1):
    try:
        start = time.time()
        response = engine.generate(prompt)
        elapsed = time.time() - start
        token_count = len(response.split())
        tps = token_count / elapsed if elapsed > 0 else 0
        
        results.append({
            'Test': f"Prompt {i}",
            'Prompt Length': len(prompt),
            'Response Length': len(response),
            'Time (s)': f"{elapsed:.2f}",
            'Tokens/sec': f"{tps:.1f}"
        })
        print(f"✓ Test {i}: {elapsed:.2f}s ({tps:.1f} tok/s)")
    except Exception as e:
        print(f"✗ Test {i}: {e}")

if results:
    results_df = pd.DataFrame(results)
    print("\n" + results_df.to_string(index=False))

## Part 7: Force Specific Backend

In [ ]:
# Force vLLM
print("Forcing vLLM backend...\n")

try:
    vllm_config = InferenceConfig(
        backend="vllm",
        model="mistral",
        temperature=0.5
    )
    vllm_engine = LocalInferenceEngine(vllm_config)
    print(f"✅ vLLM engine initialized: {vllm_engine.backend_name}")
except Exception as e:
    print(f"❌ vLLM not available: {e}")

In [ ]:
# Force Ollama
print("\nForcing Ollama backend...\n")

try:
    ollama_config = InferenceConfig(
        backend="ollama",
        model="mistral",
        base_url="http://localhost:11434"
    )
    ollama_engine = LocalInferenceEngine(ollama_config)
    print(f"✅ Ollama engine initialized: {ollama_engine.backend_name}")
except Exception as e:
    print(f"❌ Ollama not available: {e}")
    print("Ensure 'ollama serve' is running in a terminal")

## Part 8: Health Check & Diagnostics

In [ ]:
print("System Health Check\n")

health = engine.health_check()
for backend_name, status in health.items():
    available = "✅" if status.get('available') else "❌"
    healthy = "✅" if status.get('healthy') else "❌"
    
    print(f"{available} {backend_name}")
    print(f"    Healthy: {healthy}")
    if 'error' in status:
        print(f"    Error: {status['error']}")
    print()

## Part 9: Recommendations by Scenario

In [ ]:
print("📋 Scenario Recommendations\n")

scenarios = {
    "🖥️ Local Machine": {
        "Recommended": "vLLM",
        "Command": "pip install vllm",
        "Speed": "150-300 tok/s",
        "Pros": "Fast, no daemon needed"
    },
    "☁️  Google Colab": {
        "Recommended": "vLLM",
        "Command": "!pip install -q vllm",
        "Speed": "80-120 tok/s",
        "Pros": "Native support, free GPU"
    },
    "📊 Kaggle Notebooks": {
        "Recommended": "vLLM",
        "Command": "!pip install -q vllm",
        "Speed": "100-150 tok/s",
        "Pros": "Works out of the box"
    },
    "🎯 Non-Technical Users": {
        "Recommended": "Ollama",
        "Command": "ollama.ai installer",
        "Speed": "30-50 tok/s",
        "Pros": "Simple UI, downloads models automatically"
    }
}

for scenario, details in scenarios.items():
    print(f"{scenario}")
    for key, val in details.items():
        print(f"  {key}: {val}")
    print()

## Summary

✅ **Use vLLM** for:
- Maximum speed
- Cloud notebooks (Colab, Kaggle)
- Production deployments
- Any HuggingFace model

✅ **Use Ollama** for:
- Simplicity (UI + one-click setup)
- Non-technical users
- Local testing only
- When you need a REST API daemon

✅ **Use LocalInferenceEngine** for:
- Automatic backend selection
- Fallback support
- Flexible switching
- Integrated error handling